# Scenario generation

Apply an injection scenario to the grid, look at it, then write it: `scenarios/<NAME>/config.json` and the generated files in `output/<NAME>/<level>/`, one folder per network level: `nl7`, the LV connections, and `nl4`, the HV/MV substations.

The notebook is a front end to the `scengen` package, the same code as `python3 generate_scenario.py new | generate`, so the files are the same whichever way they are made. Nothing is written before section 4. The models and the file formats are described in `README.md`.

0. **Grid**: the LV networks, their connections and the region's population. Fixed inputs.
1. **Scenario**: what is assumed. Start from the SFOE table and the defaults of `config.json`, or from an existing scenario, and edit it.
2. **Allocation**: the scenario applied to the grid: nominal demand, PV and EVs per connection.
3. **Preview**: the rule and the sanity check behind `demand_factor`; then the daily product of every network, computed in memory in about a second: EV sessions, the regional day, one day per season, the energy per season and over the year (PV also at a realistic capacity factor), each network's transformer loading, one network.
4. **Write and generate**: the configuration, then `output/<NAME>/nl7/` (or `nl4/`).

Run top to bottom; after an edit in section 1, run again from there.

In [ ]:
import sys
sys.path.insert(0, ".")                  # the notebook runs in injection-scenarios/ (Jupyter and VS Code start in its folder)

import difflib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scengen.config import SCENARIOS_DIR, load_config
from scengen.demand_check import demand_check
from scengen.grid import load_grid
from scengen.injections import prepare
from scengen.models.common import SEASON_MONTHS, SEASONS
from scengen.output import output_status, write
from scengen.scenario import load_scenario, new_scenario, write_scenario

cfg = load_config()
LABEL = {"demand": "household demand", "pv": "PV", "ev": "EV charging"}
COLOUR = {"demand": "#2a78d6", "pv": "#eb6834", "ev": "#1baf7a"}          # as in the scenario browser; other streams grey
DRAW, REVERSE, NET = "#256abf", "#b8302f", "#0b0b0b"
plt.rcParams.update({"figure.figsize": (10, 4), "figure.dpi": 110, "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titlelocation": "left", "axes.titlesize": 11})


def draw_day(ax, series: dict, sign: dict, hours: np.ndarray):
    """Loads stacked upwards, generation downwards, the net load on top, on one axis."""
    for k in (1, -1):
        names = [s for s in series if sign[s] == k]
        if names:
            ax.stackplot(hours, *[k * series[s] for s in names], labels=[LABEL.get(s, s) for s in names],
                         colors=[COLOUR.get(s, "#898781") for s in names], alpha=0.4, linewidth=0)
    net = sum(sign[s] * y for s, y in series.items())
    ax.plot(hours, net, color=NET, linewidth=2, label="net load")
    ax.axhline(0, color="#898781", linewidth=0.8)
    ax.set(xlim=(hours[0], hours[-1] + hours[1] - hours[0]))
    ax.set_xticks(np.arange(0, hours[-1] + 1, 3 if len(hours) <= 96 else 12))


def day_plot(series: dict, sign: dict, hours: np.ndarray, title: str, unit: str, rating: float | None = None):
    """draw_day in a figure of its own; rating: transformer kVA as +/- lines."""
    fig, ax = plt.subplots(layout="constrained")
    draw_day(ax, series, sign, hours)
    if rating:
        for r in (rating, -rating):
            ax.axhline(r, color="#898781", linewidth=1, linestyle="--")
        ax.text(hours[0], rating, f" transformer {rating:,.0f} kVA", va="bottom", color="#52514e", fontsize=9)
    ax.set(xlabel="hour", ylabel=unit, title=title)
    fig.legend(loc="outside right upper", frameon=False)
    plt.show()


print("SFOE rows:", ", ".join(cfg["sfoe"]))
print("existing configurations:", ", ".join(sorted(p.parent.name for p in SCENARIOS_DIR.glob("*/config.json"))))

## 0. Grid

The fixed inputs, from `../data`: every connection with a fuse rating, the networks they belong to (with their transformer and the HV/MV substation feeding them) and the region's population per scenario year. A scenario does not change them.

In [ ]:
grid = load_grid()
networks = grid.networks                 # one row per network: connections, plmax_kw, trafo_kva, hvmv

print(f"{grid.name}: {len(networks):,} networks, {len(grid.connections):,} connections with a fuse rating")
print(f"fuse-rated power    {networks['plmax_kw'].sum() / 1e3:,.0f} MW, median network {networks['connections'].median():.0f} connections")
print(f"transformers        {networks['trafo_kva'].sum() / 1e3:,.0f} MVA ({networks['trafo_kva'].isna().sum()} networks without a rating)")
print(f"HV/MV substations   {networks['hvmv'].nunique()} feed {networks['hvmv'].notna().sum():,} networks ({networks['hvmv'].isna().sum()} not mapped)")
print("population          " + ", ".join(f"{n:,} in {year}" for year, n in grid.population.items()))

## 1. Scenario

`NAME` is the folder under `scenarios/` and `output/`. If `scenarios/<NAME>/config.json` exists the notebook starts from it (unless `FRESH`), otherwise from the `SFOE` row and the `scenario_defaults` of `config.json`, as `generate_scenario.py new` does.

In [ ]:
NAME = "sc1_2040"        # scenarios/NAME/config.json and output/NAME/<level>/
SFOE = "sc1_2040"        # SFOE row for a new configuration: sc1_2030, sc2_2030, sc3_2030, sc1_2040, sc2_2040, sc3_2040
FRESH = False            # True: start from the defaults even if scenarios/NAME/config.json exists

from_file = (SCENARIOS_DIR / NAME / "config.json").exists() and not FRESH
scenario = load_scenario(NAME) if from_file else new_scenario(cfg, SFOE, NAME)
print(f"{NAME}: starting from " + (f"scenarios/{NAME}/config.json (SFOE {scenario['sfoe']['key']})" if from_file else f"the defaults, SFOE {SFOE}"))

Edit the configuration here: uncomment and change what you need (blocks and units: `README.md`, *Scenario configuration*). The cell prints the result.

In [ ]:
# SFOE figures: national, scaled to the region by sfoe.to_region
# scenario["sfoe"]["to_region"] = "population"          # population: x the region's share of the national population
# scenario["sfoe"]["PV_national_MW"] *= 1.5
# scenario["sfoe"]["EV_national_thousands"] = 2940

# time
# scenario["time"]["days"] = 1                          # days of the daily product, from the representative PV day on
# scenario["time"]["yearly"]["enabled"] = False         # no hourly year per network: generation in seconds instead of minutes

# household demand
# scenario["demand"]["demand_factor"] = 0.22            # P_max of a connection / its fuse-rated power; its rule and sanity check: section 3

# PV
# scenario["pv"]["split"] = "fuse"                      # over the connections: fuse (by fuse-rated power) | connections (equally)
# scenario["pv"]["series"] = "clearsky"                 # clearsky | measured
# scenario["pv"]["season"] = "summer"                   # representative day: winter | spring | summer | autumn | all
# scenario["pv"]["scale"] = 1.0

# EV: arrival_h [h] and energy_kwh [kWh per vehicle and day], each truncated to [min, max]
# scenario["ev"]["split"] = "fuse"                      # over the connections, integers: fuse | connections
# scenario["ev"]["charger_kw"] = 11
# scenario["ev"]["arrival_h"] = {"dist": "normal", "mean": 18, "std": 0.6, "min": 17, "max": 21}
# scenario["ev"]["energy_kwh"] = {"dist": "normal", "mean": 44, "std": 0, "min": 0, "max": 100}   # the former fixed 4 h x 11 kW

for block, value in scenario.items():
    print(f"{block:7s} {json.dumps(value)}")

## 2. Allocation

The scenario applied to the grid. Each connection's nominal demand is `demand_factor` × its fuse-rated power. National PV and EVs are scaled to the region by `sfoe.to_region` (`population`: its share of the national population), then split over the connections by `pv.split` and `ev.split`: between the networks by the sum of their connections' weights and within a network by each connection's weight, the weight being its fuse-rated power (`fuse`) or the same for all (`connections`). `prepare` does it, loads the models and sets the calendar (the representative PV day); nothing is written.

In [ ]:
injections = prepare(grid, scenario)
alloc, totals, cal = injections.allocation, injections.totals, injections.calendar
streams = list(injections.streams)
sign = {s: 1 if kind == "load" else -1 for s, kind in injections.streams.items()}

facts = {"streams": ", ".join(f"{s} ({kind})" for s, kind in injections.streams.items()),
         "region share": f"{totals['region_share']:.2%} of the national figures, by {totals['to_region']} ({grid.name} in {scenario['sfoe']['year']})",
         "nominal demand": f"{alloc['nominal_demand_kw'].sum() / 1e3:,.0f} MW, {alloc['nominal_demand_kw'].mean():.1f} kW per connection",
         "PV": f"{totals['pv_allocated_kwp'] / 1e3:,.1f} MWp, {alloc['pv_kwp'].mean():.2f} kWp per connection, split by {totals['pv_split']}"}
if "ev_count" in alloc:
    facts["EVs"] = (f"{totals['ev_allocated']:,}, {alloc['ev_count'].mean():.2f} per connection, split by {totals['ev_split']}; "
                    f"{(alloc['ev_count'] > 0).mean():.0%} of the connections have at least one")
facts["daily product"] = (f"{len(cal.days)} day(s) from {cal.date}, the representative day of season "
                          f"'{scenario['pv']['season']}', every {cal.step_min} min")
facts["yearly files"] = f"every {cal.yearly_step_min} min" if cal.yearly_step_min else "no"
for k, v in facts.items():
    print(f"{k:24s} {v}")

## 3. Preview

### Household demand: the rule and the sanity check behind `demand_factor`

`demand_factor` is not an SFOE figure (reasoning: `README.md`, *Where `demand_factor` comes from*; derivation: `demand_factor.ipynb`; parameters: `demand_factor_basis` in `config.json`). `demand_check` (`scengen/demand_check.py`) computes two lines for this scenario, from a year of the networks of about the median size (a few seconds):

- **rule**: their yearly peak in % of their fuse-rated power, next to the DSO planning value that `demand_factor` is set to meet. Real networks differ somewhat from the model's median network, since their connections do not all have the same fuse rating.
- **sanity check**: the yearly energy of all connections, next to its two bounds: the households' share and the households' + services' share of Swiss consumption, prorated to the region by population. The shares only draw these bounds; they do not enter the scenario.

In [ ]:
check = demand_check(injections, cfg["demand_factor_basis"])      # config.json: planning value, Swiss consumption, sector shares
print(check)

### EV sessions

What the `ev` block draws for each vehicle and day: arrival time, energy, and the resulting charging power of the average vehicle. Each session runs at `charger_kw` from its arrival until its energy is delivered and is cut at midnight.

In [ ]:
if "ev" in injections.state:
    from scengen.models.ev import ev_node_profiles

    ev, rng, n = injections.state["ev"], np.random.default_rng(0), 20000
    arrival, energy = ev.arrival(rng, n), ev.energy(rng, n)
    pct = np.full(n, 100 / n)
    profile = ev_node_profiles(np.array([2000]), ev.charger_kw, ev.arrival, ev.energy, cal.step_min, rng)[0] / 2000

    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(12, 3.2), layout="constrained")
    a1.hist(arrival, bins=48, weights=pct, color=COLOUR["ev"])
    a1.set(xlabel="arrival [h]", ylabel="% of vehicles", title=f"arrival: mean {arrival.mean():.1f} h")
    a2.hist(energy, bins=60, weights=pct, color=COLOUR["ev"])
    a2.set(xlabel="energy [kWh]", title=f"energy: mean {energy.mean():.1f} kWh" + (" (fixed)" if energy.std() == 0 else ""))
    a3.plot(np.arange(cal.steps) * cal.step_min / 60, profile, color=COLOUR["ev"], linewidth=2)
    a3.set(xlabel="hour", ylabel="kW per vehicle", title=f"average vehicle: peak {profile.max():.2f} kW", xticks=range(0, 25, 6))
    plt.show()
    print(f"charging time: median {np.median(energy) / ev.charger_kw:.1f} h; "
          f"sessions cut at midnight: {np.mean(arrival + energy / ev.charger_kw > 24):.2%} of the vehicles")

### The region: the daily product

Every network's series summed over its connections: what section 4 writes to `daily/` (per connection) and what the screening reads.

In [ ]:
sums = injections.network_sums()                     # column (<stream>_kw, _kvar) -> [networks x days x steps]
P, Q = injections.net_load(sums)                     # per network: loads - generation, kW / kvar; P < 0 is reverse flow
hours = cal.hours

regional = {s: sums[f"{s}_kw"].sum(axis=0).ravel() / 1e3 for s in streams}
day_plot(regional, sign, hours, f"{NAME}: all {len(injections.ids):,} networks, {cal.date}", "MW")
net = P.sum(axis=0).ravel() / 1e3
dt_h = cal.step_min / 60
pd.DataFrame({"energy [MWh]": [regional[s].sum() * dt_h for s in streams] + [net.sum() * dt_h],
              "peak [MW]": [regional[s].max() for s in streams] + [net.max()],
              "at": [f"{hours[regional[s].argmax()] % 24:05.2f} h" for s in streams] + [f"{hours[net.argmax()] % 24:05.2f} h"]},
             index=[LABEL.get(s, s) for s in streams] + ["net load"]).round(1)

### The region: one day per season

The regional day on the representative day of each season: the real day closest to the season's mean PV shape, the way `pv.season` picks the day of the daily product. All four share one scale. The household demand model and the EV model have no seasons (the demand model is fitted on the days of the whole year together), so the four days differ in PV, and otherwise only in the random draws.

In [ ]:
pv = injections.state["pv"]
season_days = {q: pv.season_day(q) for q in SEASONS}       # day-of-year index of each season's representative day
season_sums = injections.network_sums(days=list(season_days.values()))     # column -> [networks x seasons x steps]
day_hours = np.arange(cal.steps) * cal.step_min / 60

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True, sharey=True, layout="constrained")
season_facts = {}
for j, (q, ax) in enumerate(zip(SEASONS, axes.flat)):
    day = {s: season_sums[f"{s}_kw"][:, j].sum(axis=0) / 1e3 for s in streams}
    draw_day(ax, day, sign, day_hours)
    date = cal.time.iloc[season_days[q] * cal.steps]
    ax.set_title(f"{q}: {date:%d %b}")
    net_day = sum(sign[s] * y for s, y in day.items())
    season_facts[f"{q}, {date:%d %b}"] = {**{f"{LABEL.get(s, s)} [MWh]": day[s].sum() * dt_h for s in streams},
                                          "net peak [MW]": net_day.max(), "lowest net [MW]": net_day.min()}
for ax in axes[1]:
    ax.set_xlabel("hour")
for ax in axes[:, 0]:
    ax.set_ylabel("MW")
fig.suptitle(f"{NAME}: all {len(injections.ids):,} networks, one day per season", x=0.02, ha="left", fontsize=11)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside right upper", frameon=False)
plt.show()
pd.DataFrame(season_facts).T.round(1)

### The region: the year

The energy of all connections per season and over the year. PV is exact: each connection's kWp times the profile. Demand and EV are simulated on every `YEAR_STRIDE`-th day and scaled to the days of each season; `YEAR_STRIDE = 1` simulates every day, as the yearly files do (about 2 minutes).

The last line puts PV at a realistic capacity factor. The profile of `pv.series` is an ideal system: south at 35°, no shading, no system losses, and for `clearsky` a year without clouds. The Swiss PV fleet as a whole yields about 954 kWh/kWp a year (2013–2022), a capacity factor of about 11 %. Sion is sunnier than the average, so treat that line as the low end. It scales PV by the same factor in every season. The value is `pv_realistic` in `config.json`, which the scenario browser reads as well; winter is December, January and February of the same year.

In [ ]:
YEAR_STRIDE = 7          # demand and EV simulated on every 7th day (about 20 s); 1 = every day, exact (about 2 min)
PV_CF = cfg["pv_realistic"]["capacity_factor"]       # realistic yearly capacity factor of PV; its source: config.json

in_season = {q: np.isin(pv.months, SEASON_MONTHS[q]) for q in SEASONS}      # day of the year -> season
sample = np.array(cal.year[::YEAR_STRIDE])
daily = {s: np.zeros(len(sample)) for s in streams}         # GWh of all connections on each simulated day
for nid in injections.ids:                                  # network by network, so that a full year needs little memory
    for column, a in injections.series(nid, sample).items():
        if column.endswith("_kw"):
            daily[column[:-3]] += a.sum(axis=(0, 2)) * dt_h / 1e6
energy = {s: np.array([daily[s][in_season[q][sample]].mean() * in_season[q].sum() for q in SEASONS]) for s in streams}
kwp, cf = alloc["pv_kwp"].sum(), pv.profile.mean()          # cf: yearly capacity factor of the profile (times pv.scale)
energy["pv"] = np.array([kwp * pv.profile[in_season[q]].sum() * dt_h / 1e6 for q in SEASONS])        # exact

rows = {LABEL.get(s, s): energy[s] for s in streams if sign[s] == 1}
rows["total load"] = load = sum(energy[s] for s in streams if sign[s] == 1)
rows[f"PV, {scenario['pv']['series']} profile (CF {cf:.1%})"] = energy["pv"]
rows[f"PV at a CF of {PV_CF:.0%}"] = energy["pv"] * PV_CF / cf
rows.update({LABEL.get(s, s): energy[s] for s in streams if sign[s] == -1 and s != "pv"})      # other generation, if any
year = pd.DataFrame(rows, index=[f"{q} [GWh]" for q in SEASONS]).T
year["year [GWh]"] = year.sum(axis=1)
print(f"PV {kwp / 1e3:,.1f} MWp: {cf * 8760:,.0f} kWh/kWp a year with the {scenario['pv']['series']} profile, {PV_CF * 8760:,.0f} "
      f"at a CF of {PV_CF:.0%}. PV energy / total load over the year: {energy['pv'].sum() / load.sum():.0%}, "
      f"{PV_CF / cf * energy['pv'].sum() / load.sum():.0%} at a CF of {PV_CF:.0%}")
year.round(1)

### The networks

Each network's transformer loading over the daily product: the largest apparent power while it draws from the MV grid, and while it feeds back (reverse flow), in % of its transformer rating, networks ranked. The screening checks voltages and line currents as well; this is the quick look.

In [ ]:
ids = injections.ids
kva = grid.networks.loc[ids, "trafo_kva"].to_numpy()
p, s = P.reshape(len(ids), -1), np.hypot(P, Q).reshape(len(ids), -1)
loading = pd.DataFrame({"draw_pct": 100 * np.where(p >= 0, s, 0).max(axis=1) / kva,
                        "reverse_pct": 100 * np.where(p < 0, s, 0).max(axis=1) / kva,
                        "connections": grid.networks.loc[ids, "connections"].to_numpy(),
                        "trafo_kva": kva}, index=pd.Index(ids, name="network_id"))

fig, ax = plt.subplots(layout="constrained")
for col, colour, text in (("draw_pct", DRAW, "peak draw"), ("reverse_pct", REVERSE, "largest reverse flow")):
    ax.plot(np.arange(1, len(loading) + 1), np.sort(loading[col].dropna().to_numpy())[::-1].tolist()
            + [np.nan] * loading[col].isna().sum(), color=colour, linewidth=2, label=text)
ax.axhline(100, color="#898781", linewidth=1, linestyle="--")
ax.set(xlabel="networks, ranked", ylabel="% of transformer rating", title=f"{NAME}: transformer loading over the daily product",
       xscale="log")
fig.legend(loc="outside right upper", frameon=False)
plt.show()
print(f"above 100 %: {(loading['draw_pct'] > 100).sum()} networks drawing, {(loading['reverse_pct'] > 100).sum()} in reverse flow, "
      f"of {len(loading):,}; no transformer rating: {loading['trafo_kva'].isna().sum()}")
loading.sort_values("reverse_pct", ascending=False).head(10).round(1)

### One network

Pick any network (default: the one with the largest reverse flow, relative to its transformer).

In [ ]:
NETWORK = loading["reverse_pct"].idxmax()        # or e.g. "LV_Grid_101"

rows, i = injections.network(NETWORK), ids.index(NETWORK)
series = {s: sums[f"{s}_kw"][i].ravel() for s in streams}
day_plot(series, sign, hours, f"{NETWORK}: {len(rows)} connections, {cal.date}", "kW", rating=loading.at[NETWORK, "trafo_kva"])
rows.drop(columns="network_id").describe().loc[["count", "mean", "min", "max"]].round(2)

## 4. Write and generate

`WRITE = True` writes the configuration to `scenarios/<NAME>/config.json`; the cell first prints how it differs from the file on disk. `GENERATE = True` writes the configuration as well and then the files of `LEVEL`, which replace `output/<NAME>/<LEVEL>/`: `nl7`, the LV connections (a `NETWORKS` subset leaves only that subset; about 3 minutes and 500 MB with the yearly files, seconds without), or `nl4`, the expected series of the HV/MV substations (every network; seconds, a few MB). The same as `generate_scenario.py generate NAME [--level nl4]`.

In [ ]:
WRITE = False            # the configuration -> scenarios/NAME/config.json
GENERATE = False         # the configuration and its files -> output/NAME/LEVEL/
LEVEL = "nl7"            # nl7: LV connections (daily/, yearly/); nl4: HV/MV substations (expected series)
NETWORKS = None          # None: every network; or a list, e.g. ["LV_Grid_101", "LV_Grid_230"] (nl7 only)

path = SCENARIOS_DIR / NAME / "config.json"
old = path.read_text().splitlines() if path.exists() else []
diff = list(difflib.unified_diff(old, json.dumps(scenario, indent=2).splitlines(), f"scenarios/{NAME}/config.json", "this notebook",
                                 lineterm="", n=1))
print("\n".join(diff) if diff else f"scenarios/{NAME}/config.json is the configuration above")
if (WRITE or GENERATE) and diff:
    print(f"-> {write_scenario(scenario)}")
if GENERATE:
    write(prepare(grid, scenario), NETWORKS, level=LEVEL)    # prepared again: the configuration as it is now, whatever was run above
print(f"output/{NAME}/{LEVEL}: {output_status(NAME, LEVEL)[1]}")

Next:

- browse the generated scenario (every stream, the hourly year, two scenarios compared): `python3 generate_scenario.py dashboard`, then `http://127.0.0.1:8050/?scenario=<NAME>`
- screen it (load flow of every network): `cd ../lv-screening && python3 -m lvgrid screen --scenario <NAME>`